# Test polytopes

## Librairies

In [7]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Go to project root (adjust depth if needed)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset

from data.mnist_data import load_mnist_datasets
from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.shortcuts.shortcut_weights import *
from src.optim.build_polytopes import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [8]:
torch.manual_seed(42)

## Large MLP (FashionMNIST)

In [9]:
# -------------------- #
# Load models and data #
# -------------------- #

# Device
device = torch.device("gpu" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# Model
model_name = "fashion_mlp_best.pth"
# model_name = "fashion_cnn_best.pth"
model_path = os.path.join(ROOT, "checkpoints", model_name)

model = FashionMLP_Large()
model.load_state_dict(torch.load(model_path))#['model_state'])
model.to(device).eval()

# qModel
qmodel = quantize_model(model, bits=4)
qmodel.to(device).eval()

# Dataset (and sample)
# Dataset (and sample)
train_dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_mlp.pt",
    weights_only=False
)

x_0, c = train_dataset[1755]
x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
x_0 = x_0.to(device) # important
print("Sample x_0 shape:", x_0.shape)
print("min and max of x_0:", x_0.min().item(), x_0.max().item())
print("Models and dataset have been loaded.")

Sample x_0 shape: torch.Size([1, 784])
min and max of x_0: -1.0 1.0
Models and dataset have been loaded.


In [10]:
# ---------------------------------------------- #
# Test single sample membership in the polytopes #
# ---------------------------------------------- #
print("*** Testing single sample in the polytopes... ***\n\n")

A_correct, b_correct, A_both, b_both = build_two_class_polytopes(model, qmodel, x_0, c)

print("A_correct shape:", A_correct.shape)
print("b_correct shape:", b_correct.shape)
print("A_both shape:", A_both.shape)
print("b_both shape:", b_both.shape)

# test A_correct x <= b_correct
x_vec = x_0.view(-1)  # shape (784,)
correct_satisfied = check_polytope_membership(A_correct, b_correct, x_0)
print("\nDoes x_0 satisfy correct_polytope constraints?", correct_satisfied.item(), "\n")

# adding bounds constraints:
# x_i <= 1  →  e_i @ x <= 1
# x_i >= -1  →  -e_i @ x <= 1
n = x_vec.shape[0]
I = torch.eye(n, dtype=A_correct.dtype)

A_bounds = torch.cat([ I, -I], dim=0).to(device)                 # (2*784, 784)
b_bounds = torch.ones(2 * n, dtype=b_correct.dtype).to(device)   # (2*784,)
b_bounds = b_bounds * -1 # flip sign of b: x ≤ 1 (or x ≥ -1) => x - 1 ≤ 0 (or x + 1 ≥ 0)

A_correct = torch.cat([A_correct, A_bounds], dim=0)    # (3849 + 1568, 784)
b_correct = torch.cat([b_correct, b_bounds], dim=0)    # (3849 + 1568,)
A_both = torch.cat([A_both, A_bounds], dim=0)    # (3849 + 1568, 784)
b_both = torch.cat([b_both, b_bounds], dim=0)    # (3849 + 1568,)

print("A_correct shape (with bounds):", A_correct.shape)
print("b_correct shape (with bounds):", b_correct.shape)
print("A_both shape (with bounds):", A_both.shape)
print("b_both shape (with bounds):", b_both.shape)

# test A_correct x <= b_correct
correct_satisfied = check_polytope_membership(A_correct, b_correct, x_0)
print("\nDoes x_0 satisfy correct_polytope constraints with bounds?", correct_satisfied.item())

*** Testing single sample in the polytopes... ***


A_correct shape: torch.Size([3849, 784])
b_correct shape: torch.Size([3849])
A_both shape: torch.Size([3858, 784])
b_both shape: torch.Size([3858])

Does x_0 satisfy correct_polytope constraints? True 

A_correct shape (with bounds): torch.Size([5417, 784])
b_correct shape (with bounds): torch.Size([5417])
A_both shape (with bounds): torch.Size([5426, 784])
b_both shape (with bounds): torch.Size([5426])

Does x_0 satisfy correct_polytope constraints with bounds? True


In [11]:
# ---------------------------------------------------------- #
# Test many samples membership in the polytopes  with bounds #
# ---------------------------------------------------------- #

for i in range(0, 10):
    
    x_0, c = train_dataset[i]
    x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
    x_0 = x_0.to(device) # important

    A_correct, b_correct, A_both, b_both = build_two_class_polytopes(model, 
                                                                     qmodel, 
                                                                     x_0, 
                                                                     c)
    
    n = x_vec.shape[0]
    I = torch.eye(n, dtype=A_correct.dtype)

    A_bounds = torch.cat([ I, -I], dim=0).to(device)                 # (2*784, 784)
    b_bounds = torch.ones(2 * n, dtype=b_correct.dtype).to(device)   # (2*784,)
    b_bounds = b_bounds * -1 # flip sign of b: x ≤ 1 (or x ≥ -1) => x - 1 ≤ 0 (or x + 1 ≥ 0)

    A_correct = torch.cat([A_correct, A_bounds], dim=0)    # (3849 + 1568, 784)
    b_correct = torch.cat([b_correct, b_bounds], dim=0)    # (3849 + 1568,)
    A_both = torch.cat([A_both, A_bounds], dim=0)    # (3849 + 1568, 784)
    b_both = torch.cat([b_both, b_bounds], dim=0)    # (3849 + 1568,)

    correct_satisfied = check_polytope_membership(A_correct, b_correct, x_0)
    both_satisfied = check_polytope_membership(A_both, b_both, x_0)
    print("Membership in correct polytope:", correct_satisfied.item(), " /  Membership in both polytopes:", both_satisfied.item())

Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True
Membership in correct polytope: True  /  Membership in both polytopes: True


In [12]:
# --------------------------------------------- #
# Test many samples membership in the polytopes #
# --------------------------------------------- #
print("\n\n*** Testing many samples in the polytopes... ***\n\n")

indices = list(range(1000))  # first 1000 samples
stats = evaluate_polytopes(model, qmodel, train_dataset, indices)

for k, v in stats.items():
    print(f"{k}: {v}")



*** Testing many samples in the polytopes... ***


total: 1000
model_correct: 1000
qmodel_correct: 991
correct_polytope_ok: 1000
both_polytope_ok: 991
